In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/xanto-carloa-z-wav/wav_files1-20241123T180758Z-002/wav_files1/wesmea/XC425095.wav
/kaggle/input/xanto-carloa-z-wav/wav_files1-20241123T180758Z-002/wav_files1/wesmea/XC355347.wav
/kaggle/input/xanto-carloa-z-wav/wav_files1-20241123T180758Z-002/wav_files1/wesmea/XC210355.wav
/kaggle/input/xanto-carloa-z-wav/wav_files1-20241123T180758Z-002/wav_files1/wesmea/XC324379.wav
/kaggle/input/xanto-carloa-z-wav/wav_files1-20241123T180758Z-002/wav_files1/wesmea/XC355345.wav
/kaggle/input/xanto-carloa-z-wav/wav_files1-20241123T180758Z-002/wav_files1/wesmea/XC310726.wav
/kaggle/input/xanto-carloa-z-wav/wav_files1-20241123T180758Z-002/wav_files1/wesmea/XC574374.wav
/kaggle/input/xanto-carloa-z-wav/wav_files1-20241123T180758Z-002/wav_files1/wesmea/XC213061.wav
/kaggle/input/xanto-carloa-z-wav/wav_files1-20241123T180758Z-002/wav_files1/wesmea/XC466361.wav
/kaggle/input/xanto-carloa-z-wav/wav_files1-20241123T180758Z-002/wav_files1/wesmea/XC539038.wav
/kaggle/input/xanto-carloa-z-wav/wav_fil

In [2]:
import os
import torch
import torchaudio
from torchaudio.transforms import MelSpectrogram, Resample
import numpy as np
from tqdm import tqdm
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset, random_split
import random
from torch.cuda.amp import autocast, GradScaler
import torch.nn.functional as F
import math
from collections import defaultdict
import matplotlib.pyplot as plt

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Set device (GPU if available, else CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

###########################################################
# Adjust paths
###########################################################

data_dirs = [
    '/kaggle/input/xanto-carloa-z-wav/wav_files-20241123T135421Z-001/wav_files',
    '/kaggle/input/xanto-carloa-z-wav/wav_files-20241123T135421Z-002/wav_files',
    '/kaggle/input/xanto-carloa-z-wav/wav_files-20241123T135421Z-003/wav_files',
    '/kaggle/input/xanto-carloa-z-wav/wav_files1-20241123T180758Z-001/wav_files1',
    '/kaggle/input/xanto-carloa-z-wav/wav_files1-20241123T180758Z-002/wav_files1'
]

model_load_path = '/kaggle/input/fastdiff-model/fastdiff_model_epoch_5.pth'

















Using device: cuda


In [3]:
###########################################################
# Data Loading (Original)
###########################################################

samples = []
labels = []
class_to_idx = {}

for root_dir in data_dirs:
    if not os.path.exists(root_dir):
        print(f"Directory {root_dir} does not exist. Skipping.")
        continue

    species_names = sorted(os.listdir(root_dir))
    for species_name in species_names:
        species_dir = os.path.join(root_dir, species_name)
        if os.path.isdir(species_dir):
            if species_name not in class_to_idx:
                class_to_idx[species_name] = len(class_to_idx)
            species_idx = class_to_idx[species_name]

            # Collect .wav files
            for file_name in os.listdir(species_dir):
                if file_name.lower().endswith('.wav'):
                    samples.append(os.path.join(species_dir, file_name))
                    labels.append(species_idx)

print(f"Total samples collected: {len(samples)}")
print(f"Total species: {len(class_to_idx)}")

idx_to_class = {v:k for k,v in class_to_idx.items()}

Total samples collected: 23784
Total species: 259


In [4]:
###########################################################
# Dataset Definition
###########################################################

class BirdSoundDataset(Dataset):
    def __init__(self, samples, labels, fixed_length=22050*4, sample_rate=22050):
        self.samples = samples
        self.labels = labels
        self.fixed_length = fixed_length
        self.sample_rate = sample_rate

        self.mel_spec_transform = MelSpectrogram(
            sample_rate=self.sample_rate,
            n_fft=1024,
            hop_length=256,
            n_mels=80
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path = self.samples[idx]
        label = self.labels[idx]

        waveform, sample_rate = torchaudio.load(file_path)

        # Resample if needed
        if sample_rate != self.sample_rate:
            resampler = Resample(orig_freq=sample_rate, new_freq=self.sample_rate)
            waveform = resampler(waveform)

        # Mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Fixed length
        if waveform.shape[1] < self.fixed_length:
            repeats = self.fixed_length // waveform.shape[1] + 1
            waveform = waveform.repeat(1, repeats)
        waveform = waveform[:, :self.fixed_length]

        # Mel-spec
        mel_spec = self.mel_spec_transform(waveform)
        mel_spec = torch.log(mel_spec + 1e-9)
        mel_spec = mel_spec.squeeze(0)

        return waveform.squeeze(0), mel_spec, label

dataset = BirdSoundDataset(samples=samples, labels=labels)


In [5]:
###########################################################
# FastDiff Model and Components
###########################################################

class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super(TimeEmbedding, self).__init__()
        self.dim=dim

    def forward(self,t):
        device=t.device
        half_dim=self.dim//2
        emb=math.log(10000)/ (half_dim-1)
        emb=torch.exp(torch.arange(half_dim,device=device)*-emb)
        emb=t.float().unsqueeze(1)*emb.unsqueeze(0)
        emb=torch.cat([torch.sin(emb),torch.cos(emb)],dim=1)
        return emb

class TimeAwareLVC(nn.Module):
    def __init__(self,in_channels,cond_channels,t_emb_dim):
        super(TimeAwareLVC,self).__init__()
        self.in_channels=in_channels
        self.kernel_predictor=nn.Sequential(
            nn.Conv1d(t_emb_dim+cond_channels,in_channels*2,kernel_size=1),
            nn.LeakyReLU(0.2),
            nn.Conv1d(in_channels*2,in_channels*2,kernel_size=1)
        )
        self.depthwise_conv=nn.Conv1d(in_channels,in_channels,kernel_size=3,padding=1,groups=in_channels)
        self.pointwise_conv=nn.Conv1d(in_channels,in_channels,kernel_size=1)

    def forward(self,x,cond,t_emb):
        t_emb_expanded=t_emb.unsqueeze(-1).expand(-1,-1,cond.size(2))
        cond_t=torch.cat([cond,t_emb_expanded],dim=1)
        kp=self.kernel_predictor(cond_t)
        kp=torch.sigmoid(kp[:,:self.in_channels,:])*torch.tanh(kp[:,self.in_channels:,:])
        x=self.depthwise_conv(x)*kp
        x=self.pointwise_conv(x)
        return x

class NoisePredictor(nn.Module):
    def __init__(self,in_channels=1,n_mel_channels=80,hidden_size=256):
        super(NoisePredictor,self).__init__()
        self.conv1=nn.Conv1d(in_channels+n_mel_channels,hidden_size,kernel_size=3,padding=1)
        self.lrelu=nn.LeakyReLU(0.2)
        self.conv2=nn.Conv1d(hidden_size,hidden_size,kernel_size=3,padding=1)
        self.fc=nn.Linear(hidden_size,1)

    def forward(self,x,mel_spec):
        x=torch.cat([x,mel_spec],dim=1)
        x=self.lrelu(self.conv1(x))
        x=self.lrelu(self.conv2(x))
        x=torch.mean(x,dim=2)
        x=self.fc(x)
        return x

class FastDiffModel(nn.Module):
    def __init__(self,residual_channels=64,cond_channels=80,t_emb_dim=128):
        super(FastDiffModel,self).__init__()
        self.input_conv=nn.Conv1d(1,residual_channels,kernel_size=7,padding=3)
        self.time_emb=TimeEmbedding(t_emb_dim)
        self.lvc_blocks=nn.ModuleList([
            TimeAwareLVC(residual_channels,cond_channels,t_emb_dim),
            TimeAwareLVC(residual_channels,cond_channels,t_emb_dim),
            TimeAwareLVC(residual_channels,cond_channels,t_emb_dim),
            TimeAwareLVC(residual_channels,cond_channels,t_emb_dim)
        ])
        self.output_conv=nn.Conv1d(residual_channels,1,kernel_size=7,padding=3)
        self.noise_predictor=NoisePredictor(in_channels=1,n_mel_channels=cond_channels)

    def forward(self,x,t,cond):
        t_emb=self.time_emb(t)
        x=self.input_conv(x)
        cond=F.interpolate(cond,size=x.size(2),mode='linear',align_corners=False)
        for lvc_block in self.lvc_blocks:
            x=x+lvc_block(x,cond,t_emb)
        x=self.output_conv(x)
        return x

    def predict_noise(self,x,cond):
        cond=F.interpolate(cond,size=x.size(2),mode='linear',align_corners=False)
        return self.noise_predictor(x,cond)

def get_noise_schedule(T,beta_start=1e-4,beta_end=0.02):
    beta_t=torch.linspace(beta_start,beta_end,T)
    alpha_t=1-beta_t
    alpha_hat_t=torch.cumprod(alpha_t,dim=0)
    return beta_t.to(device), alpha_t.to(device), alpha_hat_t.to(device)

def forward_diffusion_sample(x0,t,alpha_hat_t):
    sqrt_alpha_hat=torch.sqrt(alpha_hat_t[t]).view(-1,1,1)
    sqrt_one_minus_alpha_hat=torch.sqrt(1-alpha_hat_t[t]).view(-1,1,1)
    epsilon=torch.randn_like(x0)
    xt=sqrt_alpha_hat*x0+sqrt_one_minus_alpha_hat*epsilon
    return xt,epsilon

T=1000
beta_t,alpha_t,alpha_hat_t=get_noise_schedule(T)

model=FastDiffModel().to(device)
model.load_state_dict(torch.load(model_load_path,map_location=device))
model.eval()
print(f"Loaded FastDiff model from {model_load_path}")

Loaded FastDiff model from /kaggle/input/fastdiff-model/fastdiff_model_epoch_5.pth


/tmp/ipykernel_23/68962992.py:100: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_load_path,map_location=device))


In [ ]:
###########################################################
# Augmentation Generation
###########################################################

# For bigger improvement, let's generate more augmented samples.
# For example, pick min(5,len(indices)) original samples and generate 4 augmented per sample => 20 per species if possible.
species_to_indices=defaultdict(list)
for idx,label in enumerate(dataset.labels):
    species_to_indices[label].append(idx)

augmented_dir='augmented_data'
os.makedirs(augmented_dir, exist_ok=True)

num_original_samples_for_aug=5
num_aug_per_original=4

with torch.no_grad():
    for species_label, indices in species_to_indices.items():
        species_name=idx_to_class[species_label]
        species_dir=os.path.join(augmented_dir,f'{species_name}')
        os.makedirs(species_dir, exist_ok=True)

        if len(indices)>0:
            selected_indices=random.sample(indices, min(num_original_samples_for_aug,len(indices)))
            subset=Subset(dataset,selected_indices)
            subset_loader=DataLoader(subset,batch_size=1,shuffle=False)

            sample_count=0
            for (x0,mel_spec,label) in subset_loader:
                x0=x0.to(device)
                mel_spec=mel_spec.to(device)
                x0=x0.unsqueeze(1)
                # For each original sample, generate multiple augmented samples
                for aug_i in range(num_aug_per_original):
                    x=torch.randn_like(x0).to(device)
                    # Reverse diffusion
                    for t in reversed(range(T)):
                        t_batch=torch.tensor([t],device=device).long()
                        sqrt_alpha_t=torch.sqrt(alpha_t[t])
                        sqrt_alpha_hat_t=torch.sqrt(alpha_hat_t[t])
                        sqrt_one_minus_alpha_hat_t=torch.sqrt(1-alpha_hat_t[t])

                        t_emb=model.time_emb(t_batch).unsqueeze(-1)
                        epsilon_theta=model(x,t_batch,mel_spec)
                        if t>0:
                            beta_t_val=beta_t[t]
                            x=(1/sqrt_alpha_t)*(x-((beta_t_val/sqrt_one_minus_alpha_hat_t)*epsilon_theta))
                            noise=torch.randn_like(x)
                            x=x+torch.sqrt(beta_t_val)*noise
                        else:
                            x=(1/sqrt_alpha_t)*(x-((beta_t[t]/sqrt_one_minus_alpha_hat_t)*epsilon_theta))
                    augmented_sample=x.squeeze(1).cpu()
                    filename=os.path.join(species_dir,f'augmented_sample_{sample_count}.wav')
                    torchaudio.save(filename, augmented_sample, sample_rate=22050)

                    sample_count+=1

print("Augmentation complete. Augmented samples saved in 'augmented_data' directory.")


In [ ]:
###########################################################
# Classification Model and Training
###########################################################

class SimpleCNNClassifier(nn.Module):
    def __init__(self,n_classes):
        super(SimpleCNNClassifier,self).__init__()
        self.conv1=nn.Conv2d(1,16,kernel_size=5,stride=2,padding=2)
        self.bn1=nn.BatchNorm2d(16)
        self.conv2=nn.Conv2d(16,32,kernel_size=5,stride=2,padding=2)
        self.bn2=nn.BatchNorm2d(32)
        self.conv3=nn.Conv2d(32,64,kernel_size=5,stride=2,padding=2)
        self.bn3=nn.BatchNorm2d(64)
        self.fc=nn.Linear(64*10*5,n_classes) # Adjust if needed

    def forward(self,mel_spec):
        x=mel_spec.unsqueeze(1)
        x=self.bn1(F.relu(self.conv1(x)))
        x=self.bn2(F.relu(self.conv2(x)))
        x=self.bn3(F.relu(self.conv3(x)))
        x=x.view(x.size(0),-1)
        x=self.fc(x)
        return x

def train_classifier(model, train_loader, val_loader, n_epochs=5, lr=1e-3):
    model.to(device)
    optimizer=torch.optim.Adam(model.parameters(),lr=lr)
    criterion=nn.CrossEntropyLoss()
    for epoch in range(n_epochs):
        model.train()
        total_loss=0
        for waveform,mel_spec,label in train_loader:
            mel_spec=mel_spec.to(device)
            label=label.to(device)
            optimizer.zero_grad()
            out=model(mel_spec)
            loss=criterion(out,label)
            loss.backward()
            optimizer.step()
            total_loss+=loss.item()
        avg_loss=total_loss/len(train_loader)
        model.eval()
        correct=0
        total=0
        with torch.no_grad():
            for waveform,mel_spec,label in val_loader:
                mel_spec=mel_spec.to(device)
                label=label.to(device)
                out=model(mel_spec)
                pred=out.argmax(dim=1)
                correct+=(pred==label).sum().item()
                total+=label.size(0)
        val_acc=correct/total if total>0 else 0
        print(f"Epoch {epoch+1}/{n_epochs}, Train Loss: {avg_loss:.4f}, Val Acc: {val_acc:.4f}")

def evaluate_per_species(model, test_loader, idx_to_class):
    model.eval()
    species_correct=defaultdict(int)
    species_total=defaultdict(int)
    with torch.no_grad():
        for waveform,mel_spec,label in test_loader:
            mel_spec=mel_spec.to(device)
            label=label.to(device)
            out=model(mel_spec)
            pred=out.argmax(dim=1)
            for l,p in zip(label.cpu().tolist(), pred.cpu().tolist()):
                species_name=idx_to_class[l]
                species_total[species_name]+=1
                if p==l:
                    species_correct[species_name]+=1
    species_acc={}
    for spc in species_total:
        species_acc[spc]=species_correct[spc]/species_total[spc]
    return species_acc

    



In [ ]:
# Split original dataset into train/test
test_ratio=0.2
dataset_size=len(dataset)
test_size=int(dataset_size*test_ratio)
train_size=dataset_size - test_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))
print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

batch_size=16
train_loader_no_aug=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=batch_size,shuffle=False)

n_classes=len(class_to_idx)
clf_model=SimpleCNNClassifier(n_classes)

print("\nTraining classifier WITHOUT augmentation...")
train_classifier(clf_model, train_loader_no_aug, test_loader, n_epochs=5, lr=1e-3)
species_acc_no_aug=evaluate_per_species(clf_model,test_loader,idx_to_class)


In [ ]:
# Now load augmented data
def load_augmented_samples(aug_dir, class_to_idx):
    aug_samples=[]
    aug_labels=[]
    for spc_name, spc_idx in class_to_idx.items():
        spc_dir=os.path.join(aug_dir, spc_name)
        if os.path.isdir(spc_dir):
            for f in os.listdir(spc_dir):
                if f.lower().endswith('.wav'):
                    aug_samples.append(os.path.join(spc_dir,f))
                    aug_labels.append(spc_idx)
    return aug_samples, aug_labels

aug_samples,aug_labels=load_augmented_samples('augmented_data',class_to_idx)
print(f"Augmented samples loaded: {len(aug_samples)}")


In [ ]:
combined_samples=samples+aug_samples
combined_labels=labels+aug_labels
combined_dataset=BirdSoundDataset(samples=combined_samples, labels=combined_labels, fixed_length=22050*4)


In [ ]:
# Use same test set. We'll reconstruct train set indices:
original_train_indices=train_dataset.indices
aug_start_idx=len(dataset)
aug_end_idx=len(combined_dataset)
combined_train_indices=list(original_train_indices)
combined_train_indices.extend(range(aug_start_idx, aug_end_idx))

train_subset_with_aug=Subset(combined_dataset, combined_train_indices)
test_subset_same=test_dataset

train_loader_with_aug=DataLoader(train_subset_with_aug,batch_size=batch_size,shuffle=True)
test_loader_same=DataLoader(test_subset_same,batch_size=batch_size,shuffle=False)

clf_model_aug=SimpleCNNClassifier(n_classes)
print("\nTraining classifier WITH augmentation...")
train_classifier(clf_model_aug, train_loader_with_aug, test_loader_same, n_epochs=5, lr=1e-3)
species_acc_with_aug=evaluate_per_species(clf_model_aug,test_loader_same,idx_to_class)

In [ ]:
# Compare and print differences
print("\nPer-Species Accuracy Comparison (No Aug vs With Aug):")
print("{:<20} | {:<10} | {:<10} | {:<10}".format("Species","No Aug","With Aug","Diff"))
print("-"*60)
for spc in species_acc_no_aug:
    no_aug_acc=species_acc_no_aug[spc]
    with_aug_acc=species_acc_with_aug.get(spc, no_aug_acc)
    diff=with_aug_acc - no_aug_acc
    print("{:<20} | {:<10.4f} | {:<10.4f} | {:<+10.4f}".format(spc,no_aug_acc,with_aug_acc,diff))

In [ ]:
# Create a bar chart of difference in accuracy
species_names=list(species_acc_no_aug.keys())
diffs=[species_acc_with_aug[spc]-species_acc_no_aug[spc] for spc in species_names]

sorted_idx=np.argsort(diffs)
species_names=[species_names[i] for i in sorted_idx]
diffs=[diffs[i] for i in sorted_idx]

plt.figure(figsize=(10,6))
plt.bar(species_names, diffs, color='green', edgecolor='gray')
plt.xticks(rotation=90, fontsize=8)
plt.xlabel("Species")
plt.ylabel("Accuracy Difference (With Aug - No Aug)")
plt.title("Per-Species Accuracy Improvement with Augmentation")
plt.tight_layout()
plt.savefig("species_accuracy_diff.png", dpi=300)
plt.close()

print("\nBar chart saved as 'species_accuracy_diff.png'. End-to-end process complete.")
